In [7]:
from pathlib import Path
import sys

# Notebook liegt in /notebooks, Projektroot ist ein Level darüber
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai


In [8]:
from pathlib import Path
import json

import pandas as pd

from app.infrastructure.pdf_loader import PDFLoader

In [9]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "gesetze_im_internet"

print(f"Raw directory exists: {RAW_DIR.exists()}")
print(f"Raw directory: {RAW_DIR}")

Raw directory exists: True
Raw directory: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/raw/gesetze_im_internet


In [10]:
pdf_files = sorted(RAW_DIR.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files:")
for pdf in pdf_files:
    print(f"- {pdf.name}")

Found 3 PDF files:
- PUEG_RefE_Pflegereform_bf.pdf
- SGB_11.pdf
- SGB_5.pdf


In [11]:
loader = PDFLoader()
print("PDFLoader initialized successfully")

PDFLoader initialized successfully


In [12]:
sample_pdf = pdf_files[0] if pdf_files else None

if sample_pdf is None:
    raise FileNotFoundError(f"No PDF files found in {RAW_DIR}")

doc = loader.load(str(sample_pdf))

print("Loaded file:", sample_pdf.name)
print("Metadata:")
print(json.dumps(doc.metadata, indent=2, ensure_ascii=False))
print("\nText preview:")
print(doc.text[:1000])

TypeError: PDFLoader.load() missing 3 required keyword-only arguments: 'source', 'document_type', and 'topic'

In [ ]:
results = []

for pdf_path in pdf_files:
    try:
        loaded_doc = loader.load(str(pdf_path))
        text_len = len(loaded_doc.text)
        page_count = loaded_doc.metadata.get("page_count", 0)

        results.append(
            {
                "file_name": pdf_path.name,
                "status": "ok",
                "page_count": page_count,
                "text_length": text_len,
                "file_size_bytes": loaded_doc.metadata.get("file_size_bytes", 0),
            }
        )

    except Exception as exc:
        results.append(
            {
                "file_name": pdf_path.name,
                "status": f"error: {exc}",
                "page_count": None,
                "text_length": None,
                "file_size_bytes": None,
            }
        )

df_results = pd.DataFrame(results)
df_results

,file_name,status,page_count,text_length,file_size_bytes
0,PUEG_RefE_Pflegereform_bf.pdf,ok,112,359167,1301360
1,SGB_11.pdf,ok,177,785894,873576
2,SGB_5.pdf,ok,568,2662222,2658246


In [ ]:
if df_results.empty:
    raise ValueError("No results available. PDF loading did not run correctly.")

failed = df_results[df_results["status"] != "ok"]
if not failed.empty:
    print("Some PDFs failed to load:")
    display(failed)
else:
    print("All PDFs loaded successfully")

empty_text = df_results[(df_results["status"] == "ok") & (df_results["text_length"] == 0)]
if not empty_text.empty:
    print("Warning: Some PDFs returned empty text:")
    display(empty_text)
else:
    print("No empty-text PDFs detected")

zero_pages = df_results[(df_results["status"] == "ok") & (df_results["page_count"] == 0)]
if not zero_pages.empty:
    print("Warning: Some PDFs have 0 pages:")
    display(zero_pages)
else:
    print("No zero-page PDFs detected")

All PDFs loaded successfully
No empty-text PDFs detected
No zero-page PDFs detected


In [ ]:
for pdf_path in pdf_files:
    try:
        loaded_doc = loader.load(str(pdf_path))
        text = loaded_doc.text.strip()

        print("=" * 80)
        print(pdf_path.name)
        print(f"Characters: {len(text)}")
        print(f"Pages: {loaded_doc.metadata.get('page_count')}")
        print("Preview:")
        print(text[:500])
        print()

    except Exception as exc:
        print("=" * 80)
        print(pdf_path.name)
        print(f"ERROR: {exc}")

PUEG_RefE_Pflegereform_bf.pdf
Characters: 359167
Pages: 112
Preview:
Referentenentwurf 
des Bundesministeriums für Gesundheit 
Entwurf eines Gesetzes zur Unterstützung und Entlastung in der Pflege 
(Pflegeunterstützungs- und -entlastungsgesetz – PUEG)  
A. Problem und Ziel 
Auf der Basis von im Koalitionsvertrag für diese Legislaturperiode vorgesehenen Maßnah -
men zur Verbesserung der Situation in der Pflege sollen Anpassungen in der Pflegeversi -
cherung vorgenommen werden. Insbesondere sollen die häusliche Pflege gestärkt und pfle-
gebedürftige Menschen und ih

SGB_11.pdf
Characters: 785894
Pages: 177
Preview:
Ein Service des Bundesministerium der Justiz und für Verbraucherschutz
sowie des Bundesamts für Justiz ‒ www.gesetze-im-internet.de
Sozialgesetzbuch (SGB) - Elftes Buch (XI) - Soziale
Pflegeversicherung (Artikel 1 des Gesetzes vom 26. Mai 1994,
BGBl. I S. 1014)
SGB 11
Ausfertigungsdatum: 26.05.1994
Vollzitat:
"Das Elfte Buch Sozialgesetzbuch – Soziale Pflegeversicherung – (Art

In [ ]:
REPORT_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_path = REPORT_DIR / "sanity_check_report.csv"
df_results.to_csv(report_path, index=False)

print(f"Report saved to: {report_path}")

Report saved to: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed/sanity_check_report.csv


In [ ]:
json_report_path = REPORT_DIR / "sanity_check_report.json"
json_report_path.write_text(
    df_results.to_json(orient="records", indent=2, force_ascii=False),
    encoding="utf-8",
)

print(f"JSON report saved to: {json_report_path}")

JSON report saved to: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed/sanity_check_report.json


In [ ]:
from app.infrastructure.pdf_loader import PDFLoader
from app.repositories.document_repository import FileDocumentRepository

loader = PDFLoader()
repo = FileDocumentRepository()

pdf_path = RAW_DIR / "SGB_5.pdf"

doc = loader.load(str(pdf_path))

saved = repo.save(doc)

print("Saved:")
print(saved.document_id)

loaded = repo.read(saved.document_id)

print("\nLoaded:")
print(loaded.document_id)
print(loaded.metadata["filename"])

all_docs = repo.list_documents()

print(f"\nStored documents: {len(all_docs)}")

Saved:
d43e14a0-0b3d-40f7-afcd-2fb1a6231e1c

Loaded:
d43e14a0-0b3d-40f7-afcd-2fb1a6231e1c
SGB_5.pdf

Stored documents: 1


In [ ]:
from app.infrastructure.pdf_loader import PDFLoader
from app.repositories.document_repository import FileDocumentRepository
from app.services.document_processing_service import DocumentProcessingService

loader = PDFLoader()
repo = FileDocumentRepository()
processor = DocumentProcessingService()

pdf_path = RAW_DIR / "SGB_5.pdf"

loaded = loader.load(
    str(pdf_path),
    source="gesetze-im-internet",
    document_type="law",
    topic="krankenversicherung",
)

saved = repo.save(loaded)

chunks = processor.process(saved)

print("Document ID:", saved.document_id)
print("Metadata:", saved.metadata.to_dict())
print("Chunks:", len(chunks))
print("First chunk preview:", chunks[0].text[:500] if chunks else "No chunks")

ImportError: cannot import name 'DocumentChunk' from 'app.domain.models' (/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/app/domain/models.py)